In [1]:
from pinecone import Pinecone, ServerlessSpec    
import tqdm
from tqdm import tqdm
from embed_model import embed_text
import pandas as pd
import ast

pc = Pinecone(api_key="xxx")

c:\Users\ASUS\anaconda3\envs\ten_moi_truong\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_json("recipes_processed.json")

In [3]:
df = df.fillna("")

for col in df.columns:
    df[col] = df[col].astype(str)

df['ingredients'] = df['ingredients'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

In [4]:
df["ingredients"].iloc[0]

[{'name': 'Vịt', 'quantity': 1.0, 'unit': 'con'},
 {'name': 'Gừng', 'quantity': 100.0, 'unit': 'gr'},
 {'name': 'Rượu trắng', 'quantity': 50.0, 'unit': 'ml'},
 {'name': 'Mật ong', 'quantity': 3.0, 'unit': 'muỗng canh'},
 {'name': 'Mạch nha', 'quantity': 2.0, 'unit': 'muỗng canh'},
 {'name': 'Giấm đỏ', 'quantity': 2.0, 'unit': 'muỗng canh'},
 {'name': 'Bột tạo giòn', 'quantity': 1.0, 'unit': 'muỗng canh'},
 {'name': 'Lòng trắng trứng gà', 'quantity': 1.0, 'unit': 'quả'},
 {'name': 'Vừng trắng', 'quantity': 10.0, 'unit': 'gr'},
 {'name': 'Muối', 'quantity': 1.0, 'unit': 'ít'},
 {'name': 'Đường', 'quantity': 1.0, 'unit': 'ít'},
 {'name': 'Ngũ vị hương', 'quantity': 1.0, 'unit': 'ít'},
 {'name': 'Hành tím', 'quantity': 1.0, 'unit': 'ít'},
 {'name': 'Tỏi', 'quantity': 1.0, 'unit': 'ít'},
 {'name': 'Gừng', 'quantity': 1.0, 'unit': 'ít'},
 {'name': 'Tương hạt', 'quantity': 1.0, 'unit': 'ít'},
 {'name': 'Rượu mai quế lộ', 'quantity': 1.0, 'unit': 'ít'}]

In [5]:
def recipe_to_text(ingredients):
    text_parts = []
    for item in ingredients:
        # Lấy thông tin từ mỗi dictionary
        name = item.get('name', 'N/A')
        qty = item.get('quantity', '')
        unit = item.get('unit', '')

        # Xử lý trường hợp "ít" (không có số lượng cụ thể)
        if unit == 'ít':
            line = f"- {name}: Một ít"
        else:
            # Chuyển quantity sang số nguyên nếu nó là .0 (cho đẹp mắt)
            display_qty = int(qty) if qty % 1 == 0 else qty
            line = f"- {name}: {display_qty} {unit}"
        
        text_parts.append(line)
    
    # Nối tất cả các dòng lại bằng dấu xuống dòng
    return "Danh sách nguyên liệu:\n" + "\n".join(text_parts)

In [6]:
MAPPING_DIFFICULTY = {
    "1": "Dễ",
    "2": "Trung bình",
    "3": "Khó"
}

In [7]:
def clean_category(x):
    if not x: return ""
    try:
        val = ast.literal_eval(x) if isinstance(x, str) and x.startswith('[') else x
        return ", ".join(val) if isinstance(val, list) else val
    except:
        return x
    
df['category'] = df['category'].apply(clean_category)

In [8]:
def row_to_text(row):
    name = row["dish_name"]
    difficulty = MAPPING_DIFFICULTY.get(row["difficulty"], 0)
    ingredients = row["ingredients"]
    category = row["category"]
    prep_time = row["prep_time_min"]
    cook_time = row["cook_time_min"]
    servings = row["servings_bin"]
    popularity = row["popularity"]

    return f"""
Tên món ăn: {name}
Độ khó: {difficulty}
{recipe_to_text(ingredients)}
Thể loại: {category}
Thời gian chuẩn bị: {prep_time} phút
Thời gian nấu: {cook_time} phút
Số lượng phục vụ: {servings}
Độ phổ biến: {popularity}
"""

In [9]:
test_2 = row_to_text(df.iloc[1])
print(test_2)


Tên món ăn: Dẻ sườn bò nướng sa tế
Độ khó: Dễ
Danh sách nguyên liệu:
- Dẻ sườn bò: 500 gr
- Dừa: 1 trái
- Bơ: 30 gr
- Tỏi băm: 1 thìa canh
- Hành tây thái nhỏ: 1 thìa canh
- Tương cà: 1 thìa canh
- Tương ớt: 1 thìa canh
- Sa tế tôm: 3 thìa canh
- Muối: 0.5 thìa cà phê
- Tiêu xay: Một ít
Thể loại: Món nướng, Món từ bò
Thời gian chuẩn bị: 20 phút
Thời gian nấu: 80 phút
Số lượng phục vụ: 2-3 người
Độ phổ biến: Rất thấp



In [10]:
## HH
start = 56
end = len(df)

texts = [row_to_text(row) for _, row in df.iloc[start:end].iterrows()]
print(len(texts))
print(end)


10197
10253


In [11]:
index_name = "recipe-recommendation"

# Kiểm tra nếu index chưa tồn tại thì mới tạo
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=1536, # Thay đổi tùy theo model embedding của bạn
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws", 
            region="us-east-1" # Chọn vùng gần bạn nhất
        )
    )
    print(f"Đã tạo index: {index_name}")
else:
    print(f"Index {index_name} đã tồn tại.")

Index recipe-recommendation đã tồn tại.


In [12]:
index = pc.Index("recipe-recommendation")

In [13]:
batch_size = 16

for i in tqdm(range(0, len(texts), batch_size)):
    batch_ids = []
    batch_texts = []
    batch_names = []

    for j in range(i, min(i + batch_size, len(texts))):
        row = df.iloc[j + start]

        text = row_to_text(row)

        batch_texts.append(text)
        batch_ids.append(str(j + start))
        batch_names.append(row["dish_name"])
    
    if len(batch_texts) == 0:
        continue 

    batch_emb = embed_text(batch_texts).cpu().numpy()

    vectors = [
        {
            "id": batch_ids[k],
            "values": batch_emb[k].tolist(),
            "metadata": {
                "Tên món": batch_names[k]
            }
        }
        for k in range(len(batch_ids))
    ]

    index.upsert(vectors=vectors)

    del batch_emb, vectors

100%|██████████| 638/638 [6:01:37<00:00, 34.01s/it]  
